### Task 1 - Set up the project

Installing the needed modules.

In [1]:
# !pip install openai==1.16.2 python-dotenv pyspark

Imporint the modules

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType

from pyspark.ml.feature import VectorAssembler, PCA
from pyspark.ml.clustering import KMeans
import plotly.express as px

In [2]:
import sys

# Make sure Spark uses the same Python as your venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .master("local[*]")
         .appName("ProductRecommenderSystem")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.ui.enabled", "false")
         .getOrCreate())


In [ ]:
# import os
# os.chdir(r'C:\Users\ashle\Project\Training\IBM\guided_project\ml_pyspark_recommendersystem')

Setup the OpenAI API

In [3]:
#Loading API jey
load_dotenv("apikey.env.txt")

client = OpenAI(api_key=os.getenv("APIKEY"))

Create a Spark session

In [4]:
spark = SparkSession.builder.appName("ProductRecommenderSyster").getOrCreate()
spark

Loading the dataset

In [5]:
file_path = 'products_dataset.csv'
df = spark.read.csv(file_path, header=True, inferSchema=True, samplingRatio=1)
df.show()

+----------+--------------------+--------------------+
|product_id|               title|         description|
+----------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|
|        P1|Turmode 30 ft. RP...|If you need more ...|
|        P2|Large Tapestry Bo...|Polyester cover r...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|
|        P9|Traditional Silve...|This transitional...|
|       P10|15 in. x 59 in. O...|Its easy to add a...|
|       P11|1 qt. #350F-7 Wil...|BEHR PREMIUM PLUS...|
|       P12|Anthracite Cordle...|BlindsAvenue ligh...|
|       P13|SlimGrip 78-Inch ...|Luverne SlimGrip ...|
|       P14|6 in. x 28 in. x ...|Our Rustic Collec...|
|       P1

### Task 2 - Prepare the dataset

Combine `title` and `description` Columns

In [6]:
df = df.withColumn('combined_text', concat_ws(" ", df.title,df.description))
df.show()

+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|       combined_text|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|
|        P9|Traditional Silve...|This transitional...|Traditional Silve...|
|       P10|

get the combined_text column and convert it into a list

In [7]:
list_combined_text = df.select('combined_text').rdd.flatMap(lambda x: x).collect()
print(list_combined_text[:2])

["Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.", "Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range."]


Use OpenAI text embedding model to create the vector embeddings.

In [8]:
response = client.embeddings.create(
    input=list_combined_text,
    model="text-embedding-3-small",
    dimensions=512 # if the text is longer, can use higher dimensions
)

embedding_vectors = [data.embedding for data in response.data]
embedding_vectors[:2]  # show first 2 embedding vectors

[[0.042664770036935806,
  0.02092622220516205,
  -0.013632948510348797,
  -0.0020650296937674284,
  0.003196326084434986,
  -0.03726103901863098,
  0.027195259928703308,
  0.07833647727966309,
  0.0549556165933609,
  -0.06283164769411087,
  0.04421878606081009,
  0.048315733671188354,
  -0.0678468719124794,
  0.02521742321550846,
  0.022550875321030617,
  0.06392651796340942,
  0.10101096332073212,
  -0.030797747895121574,
  -0.08900266885757446,
  0.0706370398402214,
  -0.05802832543849945,
  0.06717582046985626,
  -0.034788742661476135,
  -0.07374507188796997,
  0.03450619429349899,
  0.04326518625020981,
  -0.05258927494287491,
  0.02505848929286003,
  0.05107057839632034,
  -0.01721777766942978,
  -0.003147762967273593,
  -0.022091733291745186,
  0.01910731941461563,
  -0.02458168938755989,
  0.04488983750343323,
  -0.06671668589115143,
  -0.02103217877447605,
  0.10447218269109726,
  -0.03870909661054611,
  0.025658903643488884,
  0.02348681539297104,
  -0.052801188081502914,
  0.

Let't put the embedding vectors into our original dataframe

Convert embedding vectors list into a Pyspark DataFrame

In [9]:
features_column_names = [f"embedding_{i}" for i in range(len(embedding_vectors[0]))]
features_column_names

['embedding_0',
 'embedding_1',
 'embedding_2',
 'embedding_3',
 'embedding_4',
 'embedding_5',
 'embedding_6',
 'embedding_7',
 'embedding_8',
 'embedding_9',
 'embedding_10',
 'embedding_11',
 'embedding_12',
 'embedding_13',
 'embedding_14',
 'embedding_15',
 'embedding_16',
 'embedding_17',
 'embedding_18',
 'embedding_19',
 'embedding_20',
 'embedding_21',
 'embedding_22',
 'embedding_23',
 'embedding_24',
 'embedding_25',
 'embedding_26',
 'embedding_27',
 'embedding_28',
 'embedding_29',
 'embedding_30',
 'embedding_31',
 'embedding_32',
 'embedding_33',
 'embedding_34',
 'embedding_35',
 'embedding_36',
 'embedding_37',
 'embedding_38',
 'embedding_39',
 'embedding_40',
 'embedding_41',
 'embedding_42',
 'embedding_43',
 'embedding_44',
 'embedding_45',
 'embedding_46',
 'embedding_47',
 'embedding_48',
 'embedding_49',
 'embedding_50',
 'embedding_51',
 'embedding_52',
 'embedding_53',
 'embedding_54',
 'embedding_55',
 'embedding_56',
 'embedding_57',
 'embedding_58',
 'embed

In [10]:
embedding_df = spark.createDataFrame(embedding_vectors, schema=features_column_names)
embedding_df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------

Add unique `row_id` to each row in the pysaprk dataframe

In [11]:
embedding_df= embedding_df.repartition(1).withColumn("id", F.monotonically_increasing_id())
embedding_df.show(2)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+-------------------

Add unique `row_id` to each row in our main pyspark dataframe `df`

In [12]:
df = df.repartition(1).withColumn("id", F.monotonically_increasing_id())
df.show(2)

+----------+--------------------+--------------------+--------------------+---+
|product_id|               title|         description|       combined_text| id|
+----------+--------------------+--------------------+--------------------+---+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|  0|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|  1|
+----------+--------------------+--------------------+--------------------+---+
only showing top 2 rows


In [13]:
df.show()

+----------+--------------------+--------------------+--------------------+---+
|product_id|               title|         description|       combined_text| id|
+----------+--------------------+--------------------+--------------------+---+
|        P0|Men's 3X Large Ca...|This heavyweight,...|Men's 3X Large Ca...|  0|
|        P1|Turmode 30 ft. RP...|If you need more ...|Turmode 30 ft. RP...|  1|
|        P2|Large Tapestry Bo...|Polyester cover r...|Large Tapestry Bo...|  2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|16-Gauge-Sinks Ve...|  3|
|        P4|Men's Crazy Horse...|This 9 in. black ...|Men's Crazy Horse...|  4|
|        P5|Mariana 6 ft. Mul...|With robust struc...|Mariana 6 ft. Mul...|  5|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|5 gal. #650C-2 Po...|  6|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|7/8 in. x 4-1/2 i...|  7|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|Ring Gold Bar Car...|  8|
|        P9|Traditional Silve...|This tr

In [14]:
embedding_df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------

Let's join the two dataframes

In [15]:
df = df.join(embedding_df, on="id", how="inner").drop("id")
df.show(2)

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+--------------------+--------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------

In [16]:
df.count()

2000

### Task 3 - Cluster products using K-means

Assemble the 512 Embedding Columns into a Single 'features' Column

- VectorAssembler is used to combine features that are going to be used in the machine learning model

In [17]:
assembler = VectorAssembler(
    inputCols=features_column_names,
    outputCol="features"
)
data = assembler.transform(df)
data = data.select("product_id", "title", "description", "features")
data.show(2)

+----------+--------------------+--------------------+--------------------+
|product_id|               title|         description|            features|
+----------+--------------------+--------------------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04266477003693...|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|
+----------+--------------------+--------------------+--------------------+
only showing top 2 rows


Apply K-Means Clustering with 5 Clusters on the `features` Column

 - Kmeans is a unsupervised machine learning technique, used to cluster the data points based on features

In [18]:
kmeans = KMeans(k=5, featuresCol='features', predictionCol='cluster')
model = kmeans.fit(data)
clustered_data = model.transform(data)
clustered_data.show()

+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|            features|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04266477003693...|      4|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|
|        P5|Mariana 6 ft. Mul...|With robust struc...|[0.06058844178915...|      0|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|[0.00749773625284...|      3|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|[-0.0217796657234...|      4|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|[-4.6415856922976...| 

### Task 4 - Visualize the clusters

Let's reduce the dimensionality of our features for visualization purpose

`512 dimensions => 2 dimensions`

- Use PCA to reduce dimensions and visualize product clusters interactively, highlighting relationships among items
- Used PySpark ML module called PCA: Principal Component Analysis
- PCA is a dimensionality reduction technique that fines the most important components of the data, making it possible to represent high dimensional data in a lower dimensional space while retaining as much information possible.

In [19]:
pca = PCA(k=2, inputCol='features', outputCol='pca_features')
pca_model = pca.fit(clustered_data)
pca_results = pca_model.transform(clustered_data)
pca_results.show()

+----------+--------------------+--------------------+--------------------+-------+--------------------+
|product_id|               title|         description|            features|cluster|        pca_features|
+----------+--------------------+--------------------+--------------------+-------+--------------------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04266477003693...|      4|[0.18866841307422...|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|[-0.1739518653657...|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|[-0.0202219742633...|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|[0.00737017697771...|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|[-0.0240411743756...|
|        P5|Mariana 6 ft. Mul...|With robust struc...|[0.06058844178915...|      0|[-0.0015003784667...|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|[

- pca_features is a vector with two-dimensions now
- features col has 512 dimensions

Convert to pandas dataframe

In [20]:
pca_df = pca_results.select("product_id", "pca_features", "cluster").toPandas()
pca_df

,product_id,pca_features,cluster
0,P0,"[0.1886684130742236, 0.03427901044484978]",4
1,P1,"[-0.1739518653657075, -0.13301099780082298]",4
2,P2,"[-0.02022197426334011, 0.316323140200805]",2
3,P3,"[0.007370176977716268, 0.059317857231531976]",0
4,P4,"[-0.02404117437560953, -0.04419558569164092]",4
...,...,...,...
1995,P1995,"[0.15093757018981876, 0.29625568679412906]",0
1996,P1996,"[-0.05670165084806678, 0.5847982511312442]",2
1997,P1997,"[0.1078274397804236, -0.004749321763927287]",1
1998,P1998,"[0.6915221979314057, 0.0969951268387874]",3


In [21]:
pca_df[['x', 'y']] = pd.DataFrame(pca_df['pca_features'].tolist(), index=pca_df.index)

In [22]:
pca_df

,product_id,pca_features,cluster,x,y
0,P0,"[0.1886684130742236, 0.03427901044484978]",4,0.188668,0.034279
1,P1,"[-0.1739518653657075, -0.13301099780082298]",4,-0.173952,-0.133011
2,P2,"[-0.02022197426334011, 0.316323140200805]",2,-0.020222,0.316323
3,P3,"[0.007370176977716268, 0.059317857231531976]",0,0.007370,0.059318
4,P4,"[-0.02404117437560953, -0.04419558569164092]",4,-0.024041,-0.044196
...,...,...,...,...,...
1995,P1995,"[0.15093757018981876, 0.29625568679412906]",0,0.150938,0.296256
1996,P1996,"[-0.05670165084806678, 0.5847982511312442]",2,-0.056702,0.584798
1997,P1997,"[0.1078274397804236, -0.004749321763927287]",1,0.107827,-0.004749
1998,P1998,"[0.6915221979314057, 0.0969951268387874]",3,0.691522,0.096995


Let's plot the Clusters

In [23]:
def plot_clusters(pca_df, num_clusters=5):
    """
    Plots a 2D visualization of clusters using Plotly Express.

    Parameters:
    - pca_df (DataFrame): A Pandas DataFrame containing columns 'x', 'y', and 'cluster'.
      'x' and 'y' are the 2D PCA components, and 'cluster' indicates the cluster label.
    - num_clusters (int): The number of unique clusters to display.
    - recently_viewed_df (DataFrame, optional): DataFrame with 'x' and 'y' coordinates for recently viewed products.

    This function creates an interactive scatter plot where each point is colored according to its cluster.
    Recently viewed products are marked as black crosses if provided.

    Returns:
    - fig (Figure): The Plotly figure object for the plot.
    """

    # Create the base cluster plot
    fig = px.scatter(
        pca_df,
        x='x',
        y='y',
        opacity=0.6,
        size_max=4,
        color= pca_df.cluster.astype(str),
        title='2D Visualization of Clusters with Recently Viewed Products',
        labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
        category_orders={'cluster': list(range(num_clusters))},
        # show the product id in the tooltip
        hover_data={'product_id': True}

    )

    # Update layout to add legend title and adjust plot settings
    fig.update_layout(legend_title_text='Clusters', legend=dict(x=1, y=1), width=600, height=500)

    return fig

fig = plot_clusters(pca_df)
fig.show()

### Task 5 - Highlight recently viewed products

List of 8 products recently viewed by the user.

In [24]:
recently_viewed_products = [
    'P316',
    'P333',
    'P1115',
    'P1691',
    'P1082',
    'P397',
    'P1441',
    'P1054',
]

In [25]:
print("The user has recently viewed the following products: ", recently_viewed_products)

The user has recently viewed the following products:  ['P316', 'P333', 'P1115', 'P1691', 'P1082', 'P397', 'P1441', 'P1054']


Let's have a look at the records in our `clustered_data` dataframe related to the recently viewed products.

In [26]:
clustered_data.show()

+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|            features|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|        P0|Men's 3X Large Ca...|This heavyweight,...|[0.04266477003693...|      4|
|        P1|Turmode 30 ft. RP...|If you need more ...|[0.04413316026329...|      4|
|        P2|Large Tapestry Bo...|Polyester cover r...|[0.04236160591244...|      2|
|        P3|16-Gauge-Sinks Ve...|It features a rec...|[-0.0497337169945...|      0|
|        P4|Men's Crazy Horse...|This 9 in. black ...|[0.02608588151633...|      4|
|        P5|Mariana 6 ft. Mul...|With robust struc...|[0.06058844178915...|      0|
|        P6|5 gal. #650C-2 Po...|BEHR PRO i300 Sem...|[0.00749773625284...|      3|
|        P7|7/8 in. x 4-1/2 i...|DEWALT High Perfo...|[-0.0217796657234...|      4|
|        P8|  Ring Gold Bar Cart|This Ring Bar Car...|[-4.6415856922976...| 

In [27]:
filtered_data = clustered_data.where(F.col("product_id").isin(recently_viewed_products))
filtered_data.show()

+----------+--------------------+--------------------+--------------------+-------+
|product_id|               title|         description|            features|cluster|
+----------+--------------------+--------------------+--------------------+-------+
|      P316|Mystic Fitz Roy B...|With its distress...|[-0.0157568920403...|      2|
|      P333|Florida Shag Beig...|Lavish natural mo...|[-0.0112434905022...|      2|
|      P397|1 gal. #M250-3 Ap...|BEHR ULTRA SCUFF ...|[-0.0059162145480...|      3|
|     P1054|1 gal. #HDPG60 Mi...|The improved PPG ...|[-0.0048829163424...|      3|
|     P1082|1 qt. #S220-7 Mol...|BEHR ULTRA SCUFF ...|[-0.0216461066156...|      3|
|     P1115|Modern Gray/Multi...|This Modern Gray/...|[-0.0226364415138...|      2|
|     P1441|1 qt. #PPU6-06 Ho...|BEHR PREMIUM PLUS...|[-0.0062980493530...|      3|
|     P1691|Genet Rust/Red-Br...|Add a refreshing ...|[-0.0304906144738...|      2|
+----------+--------------------+--------------------+--------------------+-

In [28]:
unique_clusters = filtered_data.select("cluster").distinct().rdd.flatMap(lambda x: x).collect()
print("Unique clusters from recently viewed products: ", unique_clusters)

Unique clusters from recently viewed products:  [2, 3]


### Task 6 - Recommend products based on recently viewed products

Let's have a look at the recently viewed products titles

In [29]:
filtered_data.select("title").rdd.flatMap(lambda x: x).collect()

# User was looking for rugs and interior paint & primer
# Use product recommender system to suggest similar products

["Mystic Fitz Roy Beige 9' 0 x 12' 0 Area Rug",
 'Florida Shag Beige/Multi 3 ft. x 5 ft. Floral Area Rug',
 '1 gal. #M250-3 Apple Turnover Extra Durable Flat Interior Paint & Primer',
 '1 gal. #HDPG60 Misty Emerald Lake Flat Interior Paint and Primer',
 '1 qt. #S220-7 Molasses Extra Durable Flat Interior Paint & Primer',
 'Modern Gray/Multi 9 ft. x 12 ft. Vibrant Abstract Polyester Area Rug',
 '1 qt. #PPU6-06 Honey Locust Eggshell Enamel Low Odor Interior Paint & Primer',
 'Genet Rust/Red-Brown 8 ft. x 11 ft. Abstract Wool Area Rug']

Let's see the distinct clusters of the recenetly viewed products.

In [30]:
print(unique_clusters)

[2, 3]


Let's find the possible products for the recommendation.

In [31]:
# filter only products in similar cluster
# exclude recently viewed products from recommendations

possible_recommendations = clustered_data.filter(clustered_data['cluster'].isin(unique_clusters)).filter(~clustered_data['product_id'].isin(recently_viewed_products))

Let's perform a groupby and generate a list of product IDs that can be recommended for each of the clusters.

In [32]:
recommendations = possible_recommendations.groupby("cluster").agg(F.collect_list("product_id").alias("recommendations"))
recommendations_df = recommendations.toPandas()
recommendations_df['random_recommendations'] = recommendations_df['recommendations'].apply(lambda x: np.random.choice(x, size=5, replace=False).tolist())


In [33]:
recommendations_df.head()

,cluster,recommendations,random_recommendations
0,2,"[P2, P21, P52, P71, P87, P101, P108, P119, P12...","[P538, P1318, P1721, P149, P1991]"
1,3,"[P6, P11, P16, P18, P24, P26, P30, P33, P40, P...","[P359, P692, P1968, P1276, P1750]"


In [ ]:
# write a python function to display the recommendations
def display_recommendations(row):
  # find the title of the product in df
  product_ids = row['random_recommendations']
  cluster = row.cluster

  titles = data. \
          filter(data["product_id"]. \
          isin(product_ids)).select("title").collect()

  print("\n")
  print("Recommendations for Cluster:", cluster)
  for title in titles:
    print(title[0])

recommendations_df.apply(display_recommendations, axis=1)



Recommendations for Cluster: 2
Old Farm Driftwood 8 ft. x 10 ft. Tweed Indoor/Outdoor Oval Area Rug
Lyndhurst Multi/Green 10 ft. x 14 ft. Border Area Rug
Montauk Beige/Ivory 4 ft. x 4 ft. Striped Distressed Geometric Round Area Rug
NFL - Las Vegas Raiders Black Man Cave 3 ft. x 4 ft. Area Rug
Adirondack Ivory/Silver 6 ft. x 6 ft. Square Geometric Border Area Rug


Recommendations for Cluster: 3
5 gal. #PPG1091-6 Tan Your Hide Semi-Gloss Interior Paint and Primer
1 gal. #500E-2 Aqua Breeze Eggshell Enamel Interior Stain-Blocking Paint & Primer
1 gal. #MQ5-33 Uptown Girl One-Coat Hide Semi-Gloss Enamel Interior Stain-Blocking Paint & Primer
5 gal. #760C-2 Country Beige Eggshell Interior Paint
5 gal. #HDGG31U Mountain Mist Satin Interior Paint with Primer


0    None
1    None
dtype: object

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 58727)
Traceback (most recent call last):
  File "C:\Users\ashle\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\ashle\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\ashle\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\ashle\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "c:\Users\ashle\Project\usecase\venv\Lib\site-packages\pyspark\accumulators.py", line 303, in handle
    poll(accum_updates)
  File "c:\Users\ashle\Project\usecase\venv\Lib\site-packages\pyspark\accumulators.py